# EU-27 CO2 Emissions Analysis
**Analyst:** Juan Xavier Gomez Illingworth

**NB:** Parts of the code were constructed with the assistance of Gen AI.

In [58]:
# Load EU-27 CO2 dataset and list available countries for 1999-2024
import pandas as pd
import numpy as np

url = "https://owid-public.owid.io/data/co2/owid-co2-data.csv"

data = pd.read_csv(url)
data = data[(data["year"] >= 1999) & (data["year"] <= 2024)]
data["country"].unique()

array(['Afghanistan', 'Africa', 'Africa (GCP)', 'Albania', 'Algeria',
       'Andorra', 'Angola', 'Anguilla', 'Antarctica',
       'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Asia',
       'Asia (GCP)', 'Asia (excl. China and India)', 'Australia',
       'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh',
       'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda',
       'Bhutan', 'Bolivia', 'Bonaire Sint Eustatius and Saba',
       'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde',
       'Central African Republic', 'Central America (GCP)', 'Chad',
       'Chile', 'China', 'Christmas Island', 'Colombia', 'Comoros',
       'Congo', 'Cook Islands', 'Costa Rica', "Cote d'Ivoire", 'Croatia',
       'Cuba', 'Curacao', 'Cyprus', 'Czechia',
       'Democratic Republic of Congo', 'Denmark', 'Djibouti', 'Dominica',
       'Domini

In [59]:
# Filter dataset to EU-27 records only
data = data[data["country"] == "European Union (27)"]

In [60]:
# Inspect available columns and preview the EU-27 subset
data.columns.tolist()
data.shape
data.head()

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
16242,European Union (27),1999,NaN,426830677.0,NaN,92.023933,0.215598,3589.160645,-55.488510,-1.522464,...,12.121039,12.716392,0.027536,0.107998,0.144508,0.008975,4390.518066,4028.722412,535.044067,14.907221
16243,European Union (27),2000,NaN,427627361.0,NaN,93.298416,0.218177,3600.416016,11.255296,0.313592,...,12.662358,12.639290,0.027168,0.109556,0.145829,0.009104,4337.040527,4035.282471,601.942261,16.718687
16244,European Union (27),2001,NaN,428578326.0,NaN,91.033936,0.212409,3657.100098,56.684032,1.574373,...,12.870791,12.566650,0.026817,0.111135,0.147183,0.009231,4362.858398,4079.770020,540.705811,14.785097
16245,European Union (27),2002,NaN,429890525.0,NaN,90.128769,0.209655,3657.391357,0.291328,0.007963,...,12.773079,12.489268,0.026425,0.112715,0.148495,0.009355,4351.174316,4072.231445,529.043701,14.465055
16246,European Union (27),2003,NaN,431461841.0,NaN,91.356171,0.211736,3736.358887,78.967552,2.159119,...,12.984729,12.401907,0.026032,0.114340,0.149850,0.009479,4442.458008,4146.682617,634.906128,16.992643


In [61]:
# Prepare sorted historical CO2 series for EU-27 (1999-2024)
historical_co2 = data[['year', 'co2']].copy()
historical_co2 = historical_co2.sort_values('year')
historical_co2.head()

,year,co2
16242,1999,3589.160645
16243,2000,3600.416016
16244,2001,3657.100098
16245,2002,3657.391357
16246,2003,3736.358887


In [62]:
# Fit a 2004-2024 linear trend for CO2 and project to 2050
from sklearn.linear_model import LinearRegression

data_2004_2024 = historical_co2[historical_co2['year'] >= 2004].copy()

X_train = data_2004_2024['year'].values.reshape(-1, 1)
y_train = data_2004_2024['co2'].values

model = LinearRegression()
model.fit(X_train, y_train)

trend_years = list(range(2004, 2051))
X_trend = np.array(trend_years).reshape(-1, 1)
y_trend = model.predict(X_trend)

trendline_data = pd.DataFrame({
    'year': trend_years,
    'co2_trend': y_trend
})
trendline_data[['year', 'co2_trend']].head()

,year,co2_trend
0,2004,3807.275607
1,2005,3742.990013
2,2006,3678.704419
3,2007,3614.418825
4,2008,3550.133231


In [63]:
# Define EU-27 CO2 reduction targets for 2030 (55% reduction of level in 1990) and 2050 (net zero)
target_2030 = 1914.75
target_2050 = 0

targets_df = pd.DataFrame({
    'year': [2030, 2050],
    'co2_target': [target_2030, target_2050],
    'target_label': ['2030 Target', '2050 Target']
})
targets_df

,year,co2_target,target_label
0,2030,1914.75,2030 Target
1,2050,0.00,2050 Target


In [64]:
# Compute 5-year period change statistics with gradient colors
import colorsys

periods = [
    (1999, 2004),
    (2004, 2009),
    (2009, 2014),
    (2014, 2019),
    (2019, 2024)
 ]

period_stats = []

for i, (start_year, end_year) in enumerate(periods):
    period_data = historical_co2[(historical_co2['year'] >= start_year) & 
                                 (historical_co2['year'] <= end_year)]
    
    first_value = period_data.iloc[0]['co2']
    last_value = period_data.iloc[-1]['co2']
    pct_change = ((last_value - first_value) / first_value) * 100
    
    ratio = i / (len(periods) - 1)
    
    r = 255
    g = int(255 * ratio)
    b = 0
    
    color = f"rgb({r}, {g}, {b})"
    
    period_stats.append({
        'period_start': start_year,
        'period_end': end_year,
        'period_label': f"{start_year}-{end_year}",
        'pct_change': pct_change,
        'color': color,
        'first_value': first_value,
        'last_value': last_value
    })

periods_df = pd.DataFrame(period_stats)
periods_df

,period_start,period_end,period_label,pct_change,color,first_value,last_value
0,1999,2004,1999-2004,4.296914,"rgb(255, 0, 0)",3589.160645,3743.383789
1,2004,2009,2004-2009,-11.029577,"rgb(255, 63, 0)",3743.383789,3330.504395
2,2009,2014,2009-2014,-8.756724,"rgb(255, 127, 0)",3330.504395,3038.861328
3,2014,2019,2014-2019,-4.406654,"rgb(255, 191, 0)",3038.861328,2904.949219
4,2019,2024,2019-2024,-16.481638,"rgb(255, 255, 0)",2904.949219,2426.166016


In [65]:
# Build supporting data structures for Vega-Lite chart elements
import json

historical_values = historical_co2.to_dict('records')

trendline_values = trendline_data.to_dict('records')

target_values = targets_df.to_dict('records')

rectangle_data = []

for _, row in periods_df.iterrows():
    min_co2 = 0
    max_co2 = 4000
    
    rectangle_data.append({
        'period_start': row['period_start'],
        'period_end': row['period_end'],
        'period_label': row['period_label'],
        'pct_change': row['pct_change'],
        'color': row['color'],
        'min_co2': min_co2,
        'max_co2': max_co2
    })

In [66]:
# Assemble Vega-Lite spec for EU-27 CO2 history, trendline, targets, and period shading
vega_spec = {
    "$schema": "https://vega.github.io/schema/vega-lite/v6.json",
    "config": {
        "axis": {"labelFontSize": 12, "titleFontSize": 13},
        "legend": {"labelFontSize": 12, "titleFontSize": 13},
        "title": {"fontSize": 16, "subtitleFontSize": 12}
    },
    "hconcat": [
        {
            "width": 800,
            "height": 500,
            "title": {
                "text": "EU-27 CO2 Emissions and Targets",
                "subtitle": "Historical data (1999-2024) with trendline projection to 2050 | Source: Our World In Data (https://github.com/owid/co2-data)",
                "anchor": "start"
            },
            "resolve": {
                "scale": {"x": "shared", "y": "shared"}
            },
            "layer": [
        {
            "data": {"values": rectangle_data},
            "mark": {"type": "rect", "opacity": 0.4},
            "encoding": {
                "x": {
                    "field": "period_start",
                    "type": "quantitative",
                    "scale": {"domain": [1999, 2055], "nice": False}
                },
                "x2": {"field": "period_end"},
                "y": {
                    "field": "min_co2",
                    "type": "quantitative",
                    "scale": {"domain": [0, 4000]}
                },
                "y2": {"field": "max_co2"},
                "color": {
                    "field": "color",
                    "type": "nominal",
                    "scale": None,
                    "legend": None
                },
                "tooltip": [
                    {"field": "period_label", "type": "nominal", "title": "Period"},
                    {"field": "pct_change", "type": "quantitative", "title": "Change (%)", "format": ".2f"}
                ]
            }
        },
        {
            "data": {"values": [
                {"x": (row['period_start'] + row['period_end']) / 2, "y": 400, "label": row['period_label']}
                for _, row in periods_df.iterrows()
            ]},
            "mark": {
                "type": "text",
                "angle": 270,
                "fontSize": 12,
                "baseline": "middle"
            },
            "encoding": {
                "x": {
                    "field": "x",
                    "type": "quantitative"
                },
                "y": {
                    "field": "y",
                    "type": "quantitative"
                },
                "text": {"field": "label", "type": "nominal"}
            }
        },
        {
            "data": {"values": [
                {
                    "x": (row['period_start'] + row['period_end']) / 2, 
                    "y": 1050, 
                    "pct": f"{row['pct_change']:+.1f}%",
                    "pct_change": row['pct_change']
                }
                for _, row in periods_df.iterrows()
            ]},
            "mark": {
                "type": "text",
                "angle": 270,
                "fontSize": 13,
                "fontWeight": "bold",
                "baseline": "middle"
            },
            "encoding": {
                "x": {
                    "field": "x",
                    "type": "quantitative"
                },
                "y": {
                    "field": "y",
                    "type": "quantitative"
                },
                "text": {"field": "pct", "type": "nominal"},
                "color": {
                    "condition": {
                        "test": "datum.pct_change < 0",
                        "value": "green"
                    },
                    "value": "red"
                }
            }
        },
        {
            "data": {"values": historical_values},
            "mark": {
                "type": "line",
                "point": {
                    "filled": True,
                    "size": 50,
                    "color": "black"
                },
                "color": "black",
                "strokeWidth": 2
            },
            "encoding": {
                "x": {
                    "field": "year",
                    "type": "quantitative",
                    "scale": {"domain": [1999, 2055], "nice": False},
                    "axis": {
                        "title": None, 
                        "format": "d",
                        "tickCount": 12,
                        "labelAngle": 0,
                        "grid": True
                    }
                },
                "y": {
                    "field": "co2",
                    "type": "quantitative",
                    "scale": {"domain": [0, 4000]},
                    "axis": {
                        "title": "CO2 Emissions (Million Tonnes)",
                        "grid": True
                    }
                },
                "tooltip": [
                    {"field": "year", "type": "quantitative", "title": "Year", "format": "d"},
                    {"field": "co2", "type": "quantitative", "title": "CO2 Emissions (Mt)", "format": ".2f"}
                ]
            }
        },
        {
            "data": {"values": trendline_values},
            "mark": {
                "type": "line",
                "strokeDash": [5, 5],
                "color": "gray",
                "strokeWidth": 3,
                "point": {
                    "opacity": 0,
                    "size": 200
                }
            },
            "encoding": {
                "x": {
                    "field": "year",
                    "type": "quantitative"
                },
                "y": {
                    "field": "co2_trend",
                    "type": "quantitative"
                },
                "tooltip": [
                    {"field": "year", "type": "quantitative", "title": "Year", "format": "d"},
                    {"field": "co2_trend", "type": "quantitative", "title": "Projected CO2 (Mt)", "format": ".2f"}
                ]
            }
        },
        {
            "data": {"values": target_values},
            "mark": {
                "type": "point",
                "shape": "cross",
                "size": 200,
                "filled": True
            },
            "encoding": {
                "x": {
                    "field": "year",
                    "type": "quantitative"
                },
                "y": {
                    "field": "co2_target",
                    "type": "quantitative"
                },
                "color": {
                    "condition": {
                        "test": "datum.year == 2030",
                        "value": "gold"
                    },
                    "value": "green"
                },
                "tooltip": [
                    {"field": "target_label", "type": "nominal", "title": "Target"},
                    {"field": "year", "type": "quantitative", "title": "Year", "format": "d"},
                    {"field": "co2_target", "type": "quantitative", "title": "Target CO2 (Mt)", "format": ".2f"}
                ]
            }
        }
    ]
        },
        {
            "width": 180,
            "height": 500,
            "view": {"stroke": None},
            "layer": [
                {
                    "data": {"values": [{"x": 10, "y": 450, "label": "Legend"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 15, "fontWeight": "bold"},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                },
                {
                    "data": {"values": [{"x1": 10, "x2": 40, "y": 428}]},
                    "mark": {"type": "rule", "color": "black", "strokeWidth": 2},
                    "encoding": {
                        "x": {"field": "x1", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "x2": {"field": "x2"},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None}
                    }
                },
                {
                    "data": {"values": [{"x": 45, "y": 428, "label": "Historical Data"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 12},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                },
                {
                    "data": {"values": [{"x1": 10, "x2": 40, "y": 406}]},
                    "mark": {"type": "rule", "color": "black", "strokeWidth": 2, "strokeDash": [5, 5], "opacity": 0.4},
                    "encoding": {
                        "x": {"field": "x1", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "x2": {"field": "x2"},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None}
                    }
                },
                {
                    "data": {"values": [{"x": 45, "y": 406, "label": "Trendline (2004-2024)"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 12},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                },
                {
                    "data": {"values": [{"x": 25, "y": 384}]},
                    "mark": {"type": "point", "shape": "cross", "color": "gold", "size": 150, "filled": True},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None}
                    }
                },
                {
                    "data": {"values": [{"x": 45, "y": 384, "label": "2030 Target"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 12},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                },
                {
                    "data": {"values": [{"x": 25, "y": 362}]},
                    "mark": {"type": "point", "shape": "cross", "color": "green", "size": 150, "filled": True},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None}
                    }
                },
                {
                    "data": {"values": [{"x": 45, "y": 362, "label": "2050 Target"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 12},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                }
            ]
        }
    ]
}

In [67]:
# Write CO2 targets Vega-Lite spec to JSON
output_file = "../graphs/eu_co2_emissions_targets.json"

with open(output_file, 'w') as f:
    json.dump(vega_spec, f, indent=2)

In [68]:
# Preview CO2 targets chart via Altair
import altair as alt

chart = alt.Chart.from_dict(vega_spec)
chart

alt.HConcatChart(...)

In [69]:
# Load EU-27 emissions by source and verify composition against totals
url = "https://owid-public.owid.io/data/co2/owid-co2-data.csv"

data_sources = pd.read_csv(url)
data_sources = data_sources[data_sources["country"] == "European Union (27)"]
data_sources = data_sources[data_sources["year"] >= 1999]

source_columns = [col for col in data_sources.columns if 'co2' in col.lower() and col != 'co2' and col != 'co2_per_capita' and col != 'co2_per_gdp' and col != 'co2_per_unit_energy']

sources_to_plot = ['coal_co2', 'oil_co2', 'gas_co2', 'cement_co2', 'flaring_co2', 'other_industry_co2']

data_sources['sum_sources'] = data_sources[sources_to_plot].sum(axis=1)

comparison = data_sources[['year', 'co2', 'sum_sources']].copy()
comparison['difference'] = comparison['co2'] - comparison['sum_sources']
comparison.head()

,year,co2,sum_sources,difference
16242,1999,3589.160645,3589.160910,-0.000265
16243,2000,3600.416016,3600.415701,0.000315
16244,2001,3657.100098,3657.099916,0.000181
16245,2002,3657.391357,3657.391399,-0.000042
16246,2003,3736.358887,3736.358904,-0.000017


In [70]:
# Fit source-specific linear regressions and compute zero-crossing years
from sklearn.linear_model import LinearRegression

source_info = {
    'coal_co2': {'name': 'Coal', 'color': '#8B4513'},
    'oil_co2': {'name': 'Oil', 'color': '#000000'},
    'gas_co2': {'name': 'Gas', 'color': '#4169E1'},
    'cement_co2': {'name': 'Cement', 'color': '#808080'},
    'flaring_co2': {'name': 'Flaring', 'color': '#FF4500'},
    'other_industry_co2': {'name': 'Other Industry', 'color': '#9370DB'}
}

data_2004_2024 = data_sources[(data_sources['year'] >= 2004) & (data_sources['year'] <= 2024)].copy()

projections = {}
zero_years = {}

for source_col, info in source_info.items():
    X_train = data_2004_2024['year'].values.reshape(-1, 1)
    y_train = data_2004_2024[source_col].values
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    slope = model.coef_[0]
    intercept = model.intercept_
    
    if slope < 0:
        zero_year_calc = -intercept / slope
        zero_year = int(np.ceil(zero_year_calc))
        zero_year = min(zero_year, 2130)
    else:
        zero_year = 2130
    
    proj_years = list(range(2024, zero_year + 1))
    X_proj = np.array(proj_years).reshape(-1, 1)
    y_proj = model.predict(X_proj)
    
    y_proj = np.maximum(y_proj, 0)
    
    projections[source_col] = {
        'years': proj_years,
        'values': y_proj,
        'model': model
    }
    zero_years[source_col] = zero_year
zero_years

{'coal_co2': 2041,
 'oil_co2': 2085,
 'gas_co2': 2118,
 'cement_co2': 2061,
 'flaring_co2': 2125,
 'other_industry_co2': 2087}

In [71]:
# Prepare historical, projection, and zero-crossing datasets with colors for Vega-Lite
import json

def lighten_color(hex_color, factor=0.5):
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'#{r:02x}{g:02x}{b:02x}'

for source_col, info in source_info.items():
    info['light_color'] = lighten_color(info['color'], 0.5)

historical_by_source = []
for _, row in data_sources[data_sources['year'] <= 2024].iterrows():
    for source_col, info in source_info.items():
        if pd.notna(row[source_col]):
            historical_by_source.append({
                'year': int(row['year']),
                'value': float(row[source_col]),
                'source': info['name'],
                'color': info['color'],
                'type': 'historical'
            })

projection_by_source = []
for source_col, info in source_info.items():
    proj_data = projections[source_col]
    for year, value in zip(proj_data['years'], proj_data['values']):
        projection_by_source.append({
            'year': int(year),
            'value': float(value),
            'source': info['name'],
            'color': info['light_color'],
            'type': 'projection'
        })

zero_markers = []
for source_col, info in source_info.items():
    zero_year = zero_years[source_col]
    zero_markers.append({
        'year': zero_year,
        'source': info['name'],
        'color': info['color'],
        'label': str(zero_year)
    })

In [72]:
# Assemble Vega-Lite spec for emissions by source with projections and legend
vega_spec_sources = {
    "$schema": "https://vega.github.io/schema/vega-lite/v6.json",
    "config": {
        "axis": {"labelFontSize": 12, "titleFontSize": 13},
        "legend": {"labelFontSize": 12, "titleFontSize": 13},
        "title": {"fontSize": 16, "subtitleFontSize": 12}
    },
    "hconcat": [
        {
            "width": 800,
            "height": 500,
            "title": {
                "text": "EU-27 CO2 Emissions by Source",
                "subtitle": "Historical data (1999-2024) with projections to zero emissions | Source: Our World In Data (https://github.com/owid/co2-data)",
                "anchor": "start"
            },
            "layer": [
                {
                    "data": {"values": [{"year": 2050}]},
                    "mark": {"type": "rule", "color": "green", "strokeWidth": 2},
                    "encoding": {
                        "x": {
                            "field": "year",
                            "type": "quantitative",
                            "scale": {"domain": [1999, 2130], "nice": False},
                            "axis": {
                                "title": None,
                                "format": "d",
                                "tickCount": 12,
                                "labelAngle": 0,
                                "grid": True
                            }
                        }
                    }
                },
                {
                    "data": {"values": historical_by_source},
                    "mark": {
                        "type": "line", 
                        "strokeWidth": 2,
                        "point": {
                            "filled": True,
                            "size": 100,
                            "opacity": 0
                        }
                    },
                    "encoding": {
                        "x": {
                            "field": "year",
                            "type": "quantitative",
                            "scale": {"domain": [1999, 2130], "nice": False}
                        },
                        "y": {
                            "field": "value",
                            "type": "quantitative",
                            "scale": {"domain": [0, 1700], "nice": False},
                            "axis": {
                                "title": "CO2 Emissions (Million Tonnes)",
                                "grid": True
                            }
                        },
                        "color": {
                            "field": "color",
                            "type": "nominal",
                            "scale": None,
                            "legend": None
                        },
                        "detail": {"field": "source", "type": "nominal"},
                        "tooltip": [
                            {"field": "source", "type": "nominal", "title": "Source"},
                            {"field": "year", "type": "quantitative", "title": "Year", "format": "d"},
                            {"field": "value", "type": "quantitative", "title": "CO2 (Mt)", "format": ".2f"}
                        ]
                    }
                },
                {
                    "data": {"values": projection_by_source},
                    "mark": {
                        "type": "line", 
                        "strokeWidth": 3, 
                        "strokeDash": [5, 5], 
                        "point": {
                            "filled": True,
                            "opacity": 0,
                            "size": 100
                        }
                    },
                    "encoding": {
                        "x": {
                            "field": "year",
                            "type": "quantitative"
                        },
                        "y": {
                            "field": "value",
                            "type": "quantitative"
                        },
                        "color": {
                            "field": "color",
                            "type": "nominal",
                            "scale": None,
                            "legend": None
                        },
                        "detail": {"field": "source", "type": "nominal"},
                        "tooltip": [
                            {"field": "source", "type": "nominal", "title": "Source"},
                            {"field": "year", "type": "quantitative", "title": "Year", "format": "d"},
                            {"field": "value", "type": "quantitative", "title": "Projected CO2 (Mt)", "format": ".2f"}
                        ]
                    }
                },
                {
                    "data": {"values": zero_markers},
                    "mark": {"type": "point", "size": 100, "filled": True},
                    "encoding": {
                        "x": {
                            "field": "year",
                            "type": "quantitative"
                        },
                        "y": {
                            "datum": 0,
                            "type": "quantitative"
                        },
                        "color": {
                            "field": "color",
                            "type": "nominal",
                            "scale": None,
                            "legend": None
                        },
                        "tooltip": [
                            {"field": "source", "type": "nominal", "title": "Source"},
                            {"field": "year", "type": "quantitative", "title": "Reaches Zero", "format": "d"}
                        ]
                    }
                },
                {
                    "data": {"values": zero_markers},
                    "mark": {"type": "text", "angle": 270, "fontSize": 10, "fontWeight": "bold", "baseline": "middle"},
                    "encoding": {
                        "x": {
                            "field": "year",
                            "type": "quantitative"
                        },
                        "y": {
                            "datum": 120,
                            "type": "quantitative"
                        },
                        "text": {"field": "label", "type": "nominal"},
                        "color": {
                            "field": "color",
                            "type": "nominal",
                            "scale": None,
                            "legend": None
                        }
                    }
                }
            ]
        },
        {
            "width": 180,
            "height": 500,
            "view": {"stroke": None},
            "layer": [
                {
                    "data": {"values": [{"x": 10, "y": 480, "label": "Legend"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 15, "fontWeight": "bold"},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                }
            ] + [
                item
                for i, (source_col, info) in enumerate(source_info.items())
                for item in [
                    {
                        "data": {"values": [{"x1": 10, "x2": 40, "y": 458 - i * 22}]},
                        "mark": {"type": "rule", "color": info['color'], "strokeWidth": 2},
                        "encoding": {
                            "x": {"field": "x1", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                            "x2": {"field": "x2"},
                            "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None}
                        }
                    },
                    {
                        "data": {"values": [{"x": 45, "y": 458 - i * 22, "label": info['name']}]},
                        "mark": {"type": "text", "align": "left", "fontSize": 12},
                        "encoding": {
                            "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                            "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                            "text": {"field": "label", "type": "nominal"}
                        }
                    }
                ]
            ] + [
                {
                    "data": {"values": [{"x1": 10, "x2": 40, "y": 458 - len(source_info) * 22 - 10}]},
                    "mark": {"type": "rule", "color": "green", "strokeWidth": 2},
                    "encoding": {
                        "x": {"field": "x1", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "x2": {"field": "x2"},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None}
                    }
                },
                {
                    "data": {"values": [{"x": 45, "y": 458 - len(source_info) * 22 - 10, "label": "2050 Target Year"}]},
                    "mark": {"type": "text", "align": "left", "fontSize": 12},
                    "encoding": {
                        "x": {"field": "x", "type": "quantitative", "scale": {"domain": [0, 180]}, "axis": None},
                        "y": {"field": "y", "type": "quantitative", "scale": {"domain": [0, 500]}, "axis": None},
                        "text": {"field": "label", "type": "nominal"}
                    }
                }
            ]
        }
    ]
}

In [73]:
# Save emissions-by-source Vega-Lite spec to JSON
output_file_sources = "../graphs/eu_co2_emissions_sources.json"

with open(output_file_sources, 'w') as f:
    json.dump(vega_spec_sources, f, indent=2)

In [74]:
# Preview emissions-by-source chart via Altair
import altair as alt

chart_sources = alt.Chart.from_dict(vega_spec_sources)
chart_sources

alt.HConcatChart(...)